In [0]:
# ============================================================
# 02_SILVER_CLEANING
# Purpose : Read bronze Delta tables → apply type casting,
#           null handling, bad data rules, deduplication →
#           write clean Delta tables to silver layer
#
# What this notebook does NOT do:
#   - SCD2 logic (that's 03_silver_scd2)
#   - Business aggregations (that's gold layer)
#   - Joining tables together (that's gold layer)
# ============================================================

MOUNT        = "/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project"
BRONZE_PATH  = f"{MOUNT}/bronze"
SILVER_PATH  = f"{MOUNT}/silver"

print(f"Bronze : {BRONZE_PATH}")
print(f"Silver : {SILVER_PATH}")

Bronze : /mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/bronze
Silver : /mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable

print("✓ Imports ready")

✓ Imports ready


In [0]:
# ── USERS ────────────────────────────────────────────────────
# No SCD2 needed — users don't have a changes file
# Clean job: cast types, standardise email, fix boolean

df_users_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/users")

df_users_clean = (df_users_raw

    # Cast booleans — source sends "True"/"False" strings
    # If the value is anything other than "True", treat as False
    # This is an explicit decision — document it
    .withColumn("is_active",
        F.when(F.upper(F.col("is_active")) == "TRUE", True)
         .otherwise(False)
         .cast("boolean"))

    # Standardise email — lowercase, trim whitespace
    # Reason: email matching downstream (e.g. joining to Salesforce user)
    # will silently fail if case differs
    .withColumn("email",
        F.lower(F.trim(F.col("email"))))

    # Cast dates — bronze stored as strings
    .withColumn("created_date",
        F.to_date(F.col("created_date"), "yyyy-MM-dd"))
    .withColumn("last_login_date",
        F.to_date(F.col("last_login_date"), "yyyy-MM-dd"))

    # Standardise text fields — trim whitespace
    # A single trailing space in department = broken GROUP BY in gold
    .withColumn("department",  F.trim(F.col("department")))
    .withColumn("role",        F.trim(F.col("role")))
    .withColumn("region",      F.trim(F.col("region")))
    .withColumn("first_name",  F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name",   F.initcap(F.trim(F.col("last_name"))))

    # Add silver metadata
    .withColumn("_silver_cleaned_at", F.current_timestamp())

    # Drop bronze metadata columns — silver has its own
    # We keep _source_object for traceability but drop the rest
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

# Write to silver
(df_users_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_PATH}/users"))

print(f"✓ users: {df_users_clean.count():,} rows written to silver")

✓ users: 60 rows written to silver


In [0]:
# ── PORTS ─────────────────────────────────────────────────────
# Has a _changes file — so bronze has both full + changes rows
# At silver cleaning stage we just clean ALL rows together
# SCD2 logic (which version is current) happens in 03_silver_scd2

df_ports_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/ports")

print(f"Ports bronze total (full + changes): {df_ports_raw.count():,}")
print(f"Distinct source files: {df_ports_raw.select('_source_path').distinct().count()}")

df_ports_clean = (df_ports_raw

    .withColumn("is_active",
        F.when(F.upper(F.col("is_active")) == "TRUE", True)
         .otherwise(False)
         .cast("boolean"))

    .withColumn("last_updated",
        F.to_date(F.col("last_updated"), "yyyy-MM-dd"))

    # Standardise categoricals — these become dimension attributes
    # Consistent casing prevents duplicate category values in gold
    .withColumn("port_type",
        F.initcap(F.trim(F.col("port_type"))))
    .withColumn("region",
        F.trim(F.col("region")))
    .withColumn("country_code",
        F.upper(F.trim(F.col("country_code"))))
    .withColumn("port_code",
        F.upper(F.trim(F.col("port_code"))))

    # Validate capacity — negative capacity is physically impossible
    # Decision: set invalid capacities to NULL rather than drop the row
    # Reason: the port still exists and is useful — only this one field is bad
    .withColumn("annual_capacity_teu",
        F.when(F.col("annual_capacity_teu") <= 0, None)
         .otherwise(F.col("annual_capacity_teu")))

    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

(df_ports_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_PATH}/ports_all_versions"))

# Note: folder named ports_all_versions deliberately
# After SCD2 in notebook 03, the final table will be silver/dim_ports

print(f"✓ ports: {df_ports_clean.count():,} rows written to silver/ports_all_versions")

Ports bronze total (full + changes): 18
Distinct source files: 2
✓ ports: 18 rows written to silver/ports_all_versions


In [0]:
df_ports_raw.display()

port_id,port_code,port_name,country_code,region,port_type,annual_capacity_teu,is_active,last_updated,_extracted_at,_bronze_ingested_at,_source_object,_source_path,_bronze_year,_bronze_month
PRTFF6B7E497D74,SGSIN,Port of Singapore,SG,Asia Pacific,Hub Port,2808253,True,2022-03-04,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRT55F339DB6F1E,CNSHA,Port of Shanghai,CN,Asia Pacific,Hub Port,12899271,True,2022-03-14,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRT12BAFF88754C,NLRTM,Port of Rotterdam,NL,Europe,Hub Port,5793105,True,2022-04-23,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRTB3E4413E3810,DEHAM,Port of Hamburg,DE,Europe,Hub Port,18727370,True,2022-06-30,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRT0D6017EBB1C1,USLAX,Port of Los Angeles,US,North America,Hub Port,10650790,True,2022-06-06,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRT928EA75D7AFC,AEDXB,Port of Jebel Ali,AE,Middle East,Hub Port,22443803,True,2022-05-16,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRT3DB40F71F4BA,INMAA,Port of Chennai,IN,Asia Pacific,Feeder Port,762499,True,2022-06-20,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRTF45B4EE08A91,INNSN,Nhava Sheva,IN,Asia Pacific,Feeder Port,19109364,True,2022-03-18,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRTACD67F8F1549,GBFXT,Port of Felixstowe,GB,Europe,Feeder Port,22759631,True,2022-01-27,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6
PRT731705A826C5,KRPUS,Port of Busan,KR,Asia Pacific,Hub Port,5006062,True,2022-03-09,2026-06-29T10:17:38Z,2026-06-30T12:25:41.87Z,ports,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/ports/ports_full.csv,2026,6


In [0]:
df_ports_clean.display()

port_id,port_code,port_name,country_code,region,port_type,annual_capacity_teu,is_active,last_updated,_extracted_at,_source_object,_silver_cleaned_at
PRTFF6B7E497D74,SGSIN,Port of Singapore,SG,Asia Pacific,Hub Port,2808253,true,2022-03-04,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRT55F339DB6F1E,CNSHA,Port of Shanghai,CN,Asia Pacific,Hub Port,12899271,true,2022-03-14,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRT12BAFF88754C,NLRTM,Port of Rotterdam,NL,Europe,Hub Port,5793105,true,2022-04-23,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRTB3E4413E3810,DEHAM,Port of Hamburg,DE,Europe,Hub Port,18727370,true,2022-06-30,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRT0D6017EBB1C1,USLAX,Port of Los Angeles,US,North America,Hub Port,10650790,true,2022-06-06,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRT928EA75D7AFC,AEDXB,Port of Jebel Ali,AE,Middle East,Hub Port,22443803,true,2022-05-16,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRT3DB40F71F4BA,INMAA,Port of Chennai,IN,Asia Pacific,Feeder Port,762499,true,2022-06-20,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRTF45B4EE08A91,INNSN,Nhava Sheva,IN,Asia Pacific,Feeder Port,19109364,true,2022-03-18,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRTACD67F8F1549,GBFXT,Port of Felixstowe,GB,Europe,Feeder Port,22759631,true,2022-01-27,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z
PRT731705A826C5,KRPUS,Port of Busan,KR,Asia Pacific,Hub Port,5006062,true,2022-03-09,2026-06-29T10:17:38Z,ports,2026-07-01T08:11:52.517Z


In [0]:
# ── VESSELS ───────────────────────────────────────────────────
# Same pattern as ports — has _changes file
# Clean all rows together; SCD2 in notebook 03

df_vessels_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/vessels")

print(f"Vessels bronze total (full + changes): {df_vessels_raw.count():,}")

df_vessels_clean = (df_vessels_raw

    .withColumn("is_active",
        F.when(F.upper(F.col("is_active")) == "TRUE", True)
         .otherwise(False)
         .cast("boolean"))

    .withColumn("last_updated",
        F.to_date(F.col("last_updated"), "yyyy-MM-dd"))

    # Validate year_built — sanity range check
    # No commercial vessel was built before 1950 or after current year
    # Decision: NULL out impossible values, don't drop the vessel row
    .withColumn("year_built",
        F.when(
            (F.col("year_built") < 1950) |
            (F.col("year_built") > F.year(F.current_date())),
            None)
         .otherwise(F.col("year_built")))

    # Validate capacity — must be positive
    .withColumn("capacity_teu",
        F.when(F.col("capacity_teu") <= 0, None)
         .otherwise(F.col("capacity_teu")))

    # Standardise
    .withColumn("flag_country",
        F.upper(F.trim(F.col("flag_country"))))
    .withColumn("vessel_type",
        F.initcap(F.trim(F.col("vessel_type"))))
    .withColumn("operator",
        F.trim(F.col("operator")))
    .withColumn("vessel_name",
        F.trim(F.col("vessel_name")))

    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

(df_vessels_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_PATH}/vessels_all_versions"))

print(f"✓ vessels: {df_vessels_clean.count():,} rows written to silver/vessels_all_versions")

Vessels bronze total (full + changes): 48
✓ vessels: 48 rows written to silver/vessels_all_versions


In [0]:
df_vessels_raw.display()

vessel_id,imo_number,vessel_name,vessel_type,flag_country,operator,capacity_teu,year_built,is_active,home_port_code,last_updated,_extracted_at,_bronze_ingested_at,_source_object,_source_path,_bronze_year,_bronze_month
VSLD915A4A60283,IMO8412769,Maersk Oconnell,Feeder Vessel,BS,Safmarine,22000,2018,True,SGSIN,2022-02-11,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSL8327814C4A6F,IMO1604451,Maersk Rodriguez,Container Ship,PA,MCC Transport,18000,2022,True,NLRTM,2022-05-20,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSLA9FA8ABC28C6,IMO1669324,Maersk Johnson,Container Ship,BS,MCC Transport,4500,2012,True,CNSHA,2022-04-04,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSL585E160B64F8,IMO3592974,Maersk Collins,Sub-Panamax,CY,Sealand,4500,2010,True,SGSIN,2022-06-08,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSLFE2709553460,IMO5163555,Maersk Byrd,Feeder Vessel,MH,MCC Transport,4500,2008,True,SGSIN,2022-04-16,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSL338721F58524,IMO8722606,Maersk Rubio,Panamax,LR,MCC Transport,8000,2012,True,NLRTM,2022-02-21,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSLE218CDED2EB5,IMO2164686,Maersk Sanders,Panamax,MH,MCC Transport,8000,2021,True,CNSHA,2022-03-13,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSLD24D9D0433DD,IMO3995870,Maersk Espinoza,Container Ship,SG,Safmarine,8000,2006,True,DEHAM,2022-03-08,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSL46A79D591B8F,IMO9580255,Maersk Jones,ULCV,MH,Maersk Line,12000,2023,True,SGSIN,2022-04-22,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6
VSLAE23EAA01B45,IMO4305700,Maersk Vaughn,Panamax,DK,MCC Transport,12000,2007,False,CNSHA,2022-05-14,2026-06-29T10:17:38Z,2026-06-30T12:25:43.988Z,vessels,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/vessels/vessels_full.csv,2026,6


In [0]:
df_vessels_clean.display()

vessel_id,imo_number,vessel_name,vessel_type,flag_country,operator,capacity_teu,year_built,is_active,home_port_code,last_updated,_extracted_at,_source_object,_silver_cleaned_at
VSLD915A4A60283,IMO8412769,Maersk Oconnell,Feeder Vessel,BS,Safmarine,22000,2018,true,SGSIN,2022-02-11,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSL8327814C4A6F,IMO1604451,Maersk Rodriguez,Container Ship,PA,MCC Transport,18000,2022,true,NLRTM,2022-05-20,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSLA9FA8ABC28C6,IMO1669324,Maersk Johnson,Container Ship,BS,MCC Transport,4500,2012,true,CNSHA,2022-04-04,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSL585E160B64F8,IMO3592974,Maersk Collins,Sub-panamax,CY,Sealand,4500,2010,true,SGSIN,2022-06-08,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSLFE2709553460,IMO5163555,Maersk Byrd,Feeder Vessel,MH,MCC Transport,4500,2008,true,SGSIN,2022-04-16,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSL338721F58524,IMO8722606,Maersk Rubio,Panamax,LR,MCC Transport,8000,2012,true,NLRTM,2022-02-21,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSLE218CDED2EB5,IMO2164686,Maersk Sanders,Panamax,MH,MCC Transport,8000,2021,true,CNSHA,2022-03-13,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSLD24D9D0433DD,IMO3995870,Maersk Espinoza,Container Ship,SG,Safmarine,8000,2006,true,DEHAM,2022-03-08,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSL46A79D591B8F,IMO9580255,Maersk Jones,Ulcv,MH,Maersk Line,12000,2023,true,SGSIN,2022-04-22,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z
VSLAE23EAA01B45,IMO4305700,Maersk Vaughn,Panamax,DK,MCC Transport,12000,2007,false,CNSHA,2022-05-14,2026-06-29T10:17:38Z,vessels,2026-07-01T08:23:56.14Z


In [0]:
# ── ACCOUNTS ──────────────────────────────────────────────────
# Has _changes file (tier upgrades + region reassignments)
# This is the most important dimension — drives most gold aggregations

df_accounts_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/accounts")

print(f"Accounts bronze total (full + changes): {df_accounts_raw.count():,}")

df_accounts_clean = (df_accounts_raw

    .withColumn("is_active",
        F.when(F.upper(F.col("is_active")) == "TRUE", True)
         .otherwise(False)
         .cast("boolean"))

    .withColumn("created_date",
        F.to_date(F.col("created_date"), "yyyy-MM-dd"))
    .withColumn("last_modified_date",
        F.to_date(F.col("last_modified_date"), "yyyy-MM-dd"))

    # Validate revenue — must be positive
    # Decision: NULL rather than drop — account is still valid
    .withColumn("annual_revenue_usd",
        F.when(F.col("annual_revenue_usd") <= 0, None)
         .otherwise(F.col("annual_revenue_usd")))

    # Validate employee count
    .withColumn("employee_count",
        F.when(F.col("employee_count") <= 0, None)
         .otherwise(F.col("employee_count")))

    # Standardise tier — this becomes a key filter in gold
    # "gold", "Gold", "GOLD" must all become "Gold"
    .withColumn("account_tier",
        F.initcap(F.trim(F.col("account_tier"))))
    .withColumn("region",
        F.trim(F.col("region")))
    .withColumn("country_code",
        F.upper(F.trim(F.col("country_code"))))
    .withColumn("account_name",
        F.trim(F.col("account_name")))
    .withColumn("industry",
        F.initcap(F.trim(F.col("industry"))))

    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

(df_accounts_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("region")              # accounts queried by region heavily
    .save(f"{SILVER_PATH}/accounts_all_versions"))

print(f"✓ accounts: {df_accounts_clean.count():,} rows written to silver/accounts_all_versions")

Accounts bronze total (full + changes): 340
✓ accounts: 340 rows written to silver/accounts_all_versions


In [0]:
df_accounts_raw.display()

account_id,account_name,industry,account_tier,region,country_code,annual_revenue_usd,employee_count,primary_trade_lane,owner_user_id,is_active,created_date,last_modified_date,_extracted_at,_bronze_ingested_at,_source_object,_source_path,_bronze_year,_bronze_month
ACC2D991B7A868C,"Beard, Peters and Black",Automotive,Silver,Latin America,CO,488699683,17458,JP→NL,USR9FE914DC2800,True,2022-04-26,2024-07-15,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCE074D97F96C9,Freeman-Chang,Pharmaceuticals,Silver,Latin America,PE,395349899,21173,IN→DE,USRF54A84DD8911,True,2022-01-21,2022-03-04,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACC88AF6AD82FD6,King-Martinez,Retail,Gold,Latin America,CO,297487718,26788,SG→FR,USRF4EF15C034D8,False,2022-10-10,2023-02-03,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCCC37B5DB9B39,"Washington, Hardy and Bray",FMCG,Gold,Latin America,CL,465533246,9788,MY→PL,USR19509359C4A7,True,2022-06-02,2023-12-19,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCFE31DF3E6E95,"Carlson, Hooper and Wall",Chemicals,Silver,Latin America,PE,307981523,25148,KR→SE,USR3A13C5891785,True,2022-04-07,2024-05-08,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCFCE5ADF8E73A,Wilson-Jones,Agriculture,Silver,Latin America,CL,246261856,19849,KR→SE,USR62866EA41119,True,2022-12-20,2023-04-08,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACC4289E540EC98,"Duran, Obrien and Gibbs",Technology,Gold,Latin America,CL,402300732,27280,JP→FR,USR6AA3DB66E051,True,2022-06-01,2023-12-05,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACC389A2C5A956A,Arnold and Sons,FMCG,Gold,Latin America,CO,331525766,26796,IN→DE,USR54E7D19D5D69,True,2022-02-20,2023-05-10,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCCE4444B7E6B8,"Walker, Hernandez and Baker",Retail,Silver,Latin America,BR,391435609,43231,IN→PL,USR122BEA281760,True,2022-11-11,2022-12-12,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCFB5ADC4A3E61,Larson-Holloway,Technology,Bronze,Latin America,CO,205496488,39200,IN→PL,USR6EEE6E5AAA58,True,2022-11-16,2024-01-30,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6


In [0]:
df_accounts_clean.display()

account_id,account_name,industry,account_tier,region,country_code,annual_revenue_usd,employee_count,primary_trade_lane,owner_user_id,is_active,created_date,last_modified_date,_extracted_at,_source_object,_silver_cleaned_at
ACC2D991B7A868C,"Beard, Peters and Black",Automotive,Silver,Latin America,CO,488699683,17458,JP→NL,USR9FE914DC2800,true,2022-04-26,2024-07-15,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACCE074D97F96C9,Freeman-Chang,Pharmaceuticals,Silver,Latin America,PE,395349899,21173,IN→DE,USRF54A84DD8911,true,2022-01-21,2022-03-04,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACC88AF6AD82FD6,King-Martinez,Retail,Gold,Latin America,CO,297487718,26788,SG→FR,USRF4EF15C034D8,false,2022-10-10,2023-02-03,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACCCC37B5DB9B39,"Washington, Hardy and Bray",Fmcg,Gold,Latin America,CL,465533246,9788,MY→PL,USR19509359C4A7,true,2022-06-02,2023-12-19,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACCFE31DF3E6E95,"Carlson, Hooper and Wall",Chemicals,Silver,Latin America,PE,307981523,25148,KR→SE,USR3A13C5891785,true,2022-04-07,2024-05-08,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACCFCE5ADF8E73A,Wilson-Jones,Agriculture,Silver,Latin America,CL,246261856,19849,KR→SE,USR62866EA41119,true,2022-12-20,2023-04-08,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACC4289E540EC98,"Duran, Obrien and Gibbs",Technology,Gold,Latin America,CL,402300732,27280,JP→FR,USR6AA3DB66E051,true,2022-06-01,2023-12-05,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACC389A2C5A956A,Arnold and Sons,Fmcg,Gold,Latin America,CO,331525766,26796,IN→DE,USR54E7D19D5D69,true,2022-02-20,2023-05-10,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACCCE4444B7E6B8,"Walker, Hernandez and Baker",Retail,Silver,Latin America,BR,391435609,43231,IN→PL,USR122BEA281760,true,2022-11-11,2022-12-12,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z
ACCFB5ADC4A3E61,Larson-Holloway,Technology,Bronze,Latin America,CO,205496488,39200,IN→PL,USR6EEE6E5AAA58,true,2022-11-16,2024-01-30,2026-06-29T10:17:38Z,accounts,2026-07-01T08:47:12.47Z


In [0]:
# ── CONTACTS ──────────────────────────────────────────────────
df_contacts_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/contacts")

df_contacts_clean = (df_contacts_raw
    .withColumn("is_active",
        F.when(F.upper(F.col("is_active")) == "TRUE", True)
         .otherwise(False)
         .cast("boolean"))
    .withColumn("created_date",
        F.to_date(F.col("created_date"), "yyyy-MM-dd"))
    .withColumn("email",
        F.lower(F.trim(F.col("email"))))
    .withColumn("first_name",
        F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name",
        F.initcap(F.trim(F.col("last_name"))))
    .withColumn("job_title",
        F.initcap(F.trim(F.col("job_title"))))
    .withColumn("contact_type",
        F.initcap(F.trim(F.col("contact_type"))))
    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

(df_contacts_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_PATH}/contacts"))

print(f"✓ contacts: {df_contacts_clean.count():,} rows written to silver")

# ── CONTRACTS ─────────────────────────────────────────────────
df_contracts_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/contracts")

df_contracts_clean = (df_contracts_raw
    .withColumn("start_date",
        F.to_date(F.col("start_date"), "yyyy-MM-dd"))
    .withColumn("end_date",
        F.to_date(F.col("end_date"), "yyyy-MM-dd"))
    .withColumn("created_date",
        F.to_date(F.col("created_date"), "yyyy-MM-dd"))

    # Validate contract value and rates
    .withColumn("contract_value_usd",
        F.when(F.col("contract_value_usd") <= 0, None)
         .otherwise(F.col("contract_value_usd")))
    .withColumn("rate_per_teu_usd",
        F.when(F.col("rate_per_teu_usd") <= 0, None)
         .otherwise(F.col("rate_per_teu_usd")))
    .withColumn("volume_commitment_teu",
        F.when(F.col("volume_commitment_teu") <= 0, None)
         .otherwise(F.col("volume_commitment_teu")))

    # Validate date logic — end must be after start
    # Decision: NULL end_date if invalid rather than drop contract
    .withColumn("end_date",
        F.when(F.col("end_date") <= F.col("start_date"), None)
         .otherwise(F.col("end_date")))

    .withColumn("status",
        F.initcap(F.trim(F.col("status"))))
    .withColumn("contract_type",
        F.initcap(F.trim(F.col("contract_type"))))
    .withColumn("rate_tier",
        F.initcap(F.trim(F.col("rate_tier"))))
    .withColumn("cargo_type",
        F.initcap(F.trim(F.col("cargo_type"))))

    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

(df_contracts_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("status")
    .save(f"{SILVER_PATH}/contracts"))

print(f"✓ contracts: {df_contracts_clean.count():,} rows written to silver")

✓ contacts: 600 rows written to silver
✓ contracts: 500 rows written to silver


In [0]:
# ── BOOKINGS ──────────────────────────────────────────────────
# This is the core fact source — deserves the most care
# Three real data quality problems to handle:
#   1. Null cargo_type (~3% of rows)
#   2. Negative teu_count (~2% of rows)
#   3. Missing actual_departure / actual_arrival (expected — voyage not complete)

df_bookings_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/bookings")

total_raw = df_bookings_raw.count()
print(f"Raw bookings from bronze: {total_raw:,}")

# Profile the issues BEFORE cleaning
# This is what you'd do in a real project to understand the data
null_cargo    = df_bookings_raw.filter(F.col("cargo_type").isNull()).count()
negative_teu  = df_bookings_raw.filter(F.col("teu_count") < 0).count()
null_actual_dep = df_bookings_raw.filter(F.col("actual_departure").isNull()).count()

print(f"\nData quality profile (pre-cleaning):")
print(f"  Null cargo_type      : {null_cargo:,}  ({100*null_cargo/total_raw:.1f}%)")
print(f"  Negative teu_count   : {negative_teu:,}  ({100*negative_teu/total_raw:.1f}%)")
print(f"  Null actual_departure: {null_actual_dep:,}  ({100*null_actual_dep/total_raw:.1f}%)")

Raw bookings from bronze: 2,000

Data quality profile (pre-cleaning):
  Null cargo_type      : 56  (2.8%)
  Negative teu_count   : 37  (1.9%)
  Null actual_departure: 287  (14.3%)


In [0]:
df_bookings_clean = (df_bookings_raw

    # ── Date casts ────────────────────────────────────────────
    .withColumn("booking_date",
        F.to_timestamp(F.col("booking_date"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("etd",
        F.to_date(F.col("etd"), "yyyy-MM-dd"))
    .withColumn("eta",
        F.to_date(F.col("eta"), "yyyy-MM-dd"))
    .withColumn("actual_departure",
        F.to_date(F.col("actual_departure"), "yyyy-MM-dd"))
    .withColumn("actual_arrival",
        F.to_date(F.col("actual_arrival"), "yyyy-MM-dd"))
    .withColumn("created_date",
        F.to_date(F.col("created_date"), "yyyy-MM-dd"))
    .withColumn("last_modified_date",
        F.to_date(F.col("last_modified_date"), "yyyy-MM-dd"))

    # ── Handle null cargo_type ────────────────────────────────
    # Decision: replace with "Unknown" NOT drop the row
    # Reason: the booking still happened and has revenue
    # Dropping it would undercount total revenue in gold
    # "Unknown" is visible in reports — prompts investigation at source
    # The WRONG decision would be to impute a category (guess) — that
    # would make the data look clean while being factually wrong
    .withColumn("cargo_type",
        F.coalesce(
            F.initcap(F.trim(F.col("cargo_type"))),
            F.lit("Unknown")))

    # ── Handle negative teu_count ─────────────────────────────
    # Decision: take absolute value (abs) NOT drop or NULL
    # Reason: negative TEU is almost certainly a data entry error
    # (minus sign accidentally added). The magnitude is likely correct.
    # We flag it with a new column so analysts can filter if needed
    .withColumn("teu_count_is_corrected",
        F.when(F.col("teu_count") < 0, True).otherwise(False))
    .withColumn("teu_count",
        F.abs(F.col("teu_count")))

    # ── Validate ETD/ETA logic ────────────────────────────────
    # ETA must be after ETD — flag invalid records
    .withColumn("has_invalid_dates",
        F.when(
            F.col("eta").isNotNull() &
            F.col("etd").isNotNull() &
            (F.col("eta") <= F.col("etd")),
            True).otherwise(False))

    # ── Validate revenue ──────────────────────────────────────
    # Negative revenue is not correctable — NULL it
    # A booking with negative revenue needs source investigation
    .withColumn("total_revenue_usd",
        F.when(F.col("total_revenue_usd") < 0, None)
         .otherwise(F.col("total_revenue_usd")))

    # ── Derive booking_month for partitioning ─────────────────
    # Bookings are almost always filtered by date in gold queries
    # Partitioning by booking_month makes those queries cheap
    .withColumn("booking_year",
        F.year(F.col("booking_date")))
    .withColumn("booking_month",
        F.month(F.col("booking_date")))

    # ── Standardise categoricals ──────────────────────────────
    .withColumn("booking_status",
        F.initcap(F.trim(F.col("booking_status"))))
    .withColumn("container_size",
        F.upper(F.trim(F.col("container_size"))))

    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

# Write partitioned by year + month — most gold queries filter on booking date
(df_bookings_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("booking_year", "booking_month")
    .save(f"{SILVER_PATH}/bookings"))

# Post-cleaning profile — confirms decisions were applied
total_clean       = df_bookings_clean.count()
null_cargo_after  = df_bookings_clean.filter(F.col("cargo_type") == "Unknown").count()
corrected_teu     = df_bookings_clean.filter(F.col("teu_count_is_corrected") == True).count()
invalid_dates     = df_bookings_clean.filter(F.col("has_invalid_dates") == True).count()

print(f"\nData quality profile (post-cleaning):")
print(f"  Total rows           : {total_clean:,}")
print(f"  cargo_type='Unknown' : {null_cargo_after:,}  (was null, now visible)")
print(f"  teu_count corrected  : {corrected_teu:,}  (abs value applied, flagged)")
print(f"  Invalid ETD/ETA      : {invalid_dates:,}  (flagged for investigation)")
print(f"\n✓ bookings written to silver — partitioned by booking_year, booking_month")


Data quality profile (post-cleaning):
  Total rows           : 2,000
  cargo_type='Unknown' : 56  (was null, now visible)
  teu_count corrected  : 37  (abs value applied, flagged)
  Invalid ETD/ETA      : 0  (flagged for investigation)

✓ bookings written to silver — partitioned by booking_year, booking_month


In [0]:
# ── CARGO EVENTS ──────────────────────────────────────────────
df_events_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/cargo_events")

df_events_clean = (df_events_raw
    .withColumn("event_timestamp",
        F.to_timestamp(F.col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))

    # Validate coordinates
    .withColumn("location_lat",
        F.when(
            (F.col("location_lat") < -90) |
            (F.col("location_lat") > 90),
            None).otherwise(F.col("location_lat")))
    .withColumn("location_lon",
        F.when(
            (F.col("location_lon") < -180) |
            (F.col("location_lon") > 180),
            None).otherwise(F.col("location_lon")))

    # Validate delay hours — must be non-negative
    .withColumn("delay_hours",
        F.when(F.col("delay_hours") < 0, 0)
         .otherwise(F.col("delay_hours")))

    .withColumn("event_type",
        F.initcap(F.trim(F.col("event_type"))))

    # Derive event date for partitioning
    .withColumn("event_date",
        F.to_date(F.col("event_timestamp")))

    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

(df_events_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("event_date")
    .save(f"{SILVER_PATH}/cargo_events"))

print(f"✓ cargo_events: {df_events_clean.count():,} rows written to silver")

# ── CASES ─────────────────────────────────────────────────────
df_cases_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/cases")

df_cases_clean = (df_cases_raw
    .withColumn("created_date",
        F.to_date(F.col("created_date"), "yyyy-MM-dd"))
    .withColumn("resolved_date",
        F.to_date(F.col("resolved_date"), "yyyy-MM-dd"))

    # Validate resolution_days — must be non-negative
    # Negative = resolved before created = impossible
    .withColumn("resolution_days",
        F.when(F.col("resolution_days") < 0, None)
         .otherwise(F.col("resolution_days")))

    # Validate customer_rating — must be 1–5
    .withColumn("customer_rating",
        F.when(
            (F.col("customer_rating") < 1) |
            (F.col("customer_rating") > 5),
            None).otherwise(F.col("customer_rating")))

    .withColumn("status",   F.initcap(F.trim(F.col("status"))))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("priority", F.initcap(F.trim(F.col("priority"))))

    .withColumn("_silver_cleaned_at", F.current_timestamp())
    .drop("_bronze_ingested_at", "_source_path",
          "_bronze_year", "_bronze_month")
)

(df_cases_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("status")
    .save(f"{SILVER_PATH}/cases"))

print(f"✓ cases: {df_cases_clean.count():,} rows written to silver")

✓ cargo_events: 7,115 rows written to silver
✓ cases: 800 rows written to silver


In [0]:
# ── Final summary across all silver tables ───────────────────
silver_tables = [
    "users", "ports_all_versions", "vessels_all_versions",
    "accounts_all_versions", "contacts", "contracts",
    "bookings", "cargo_events", "cases"
]

print("SILVER CLEANING SUMMARY")
print("="*55)
print(f"{'Table':<30} {'Rows':>10}  {'Status'}")
print("-"*55)

for table in silver_tables:
    try:
        df = spark.read.format("delta").load(f"{SILVER_PATH}/{table}")
        count = df.count()
        print(f"{table:<30} {count:>10,}  ✓")
    except Exception as e:
        print(f"{table:<30} {'—':>10}   ✗ {str(e)[:40]}")

print("="*55)
print("""
WHAT HAPPENS NEXT:
  03_silver_scd2  → process ports_all_versions, vessels_all_versions,
                    accounts_all_versions into proper SCD2 dimensions
                    with valid_from, valid_to, is_current columns
""")

SILVER CLEANING SUMMARY
Table                                Rows  Status
-------------------------------------------------------
users                                  60  ✓
ports_all_versions                     18  ✓
vessels_all_versions                   48  ✓
accounts_all_versions                 340  ✓
contacts                              600  ✓
contracts                             500  ✓
bookings                            2,000  ✓
cargo_events                        7,115  ✓
cases                                 800  ✓

WHAT HAPPENS NEXT:
  03_silver_scd2  → process ports_all_versions, vessels_all_versions,
                    accounts_all_versions into proper SCD2 dimensions
                    with valid_from, valid_to, is_current columns



In [0]:
print("""
EXPORT THIS NOTEBOOK:
  File → Export → Source File (.py)
  Save to: notebooks/02_silver_cleaning.py

THEN PUSH TO GITHUB:
  git add notebooks/02_silver_cleaning.py
  git commit -m "Add silver cleaning notebook — type casting, null handling, bad data rules"
  git push origin main
""")


EXPORT THIS NOTEBOOK:
  File → Export → Source File (.py)
  Save to: notebooks/02_silver_cleaning.py

THEN PUSH TO GITHUB:
  git add notebooks/02_silver_cleaning.py
  git commit -m "Add silver cleaning notebook — type casting, null handling, bad data rules"
  git push origin main

